# ITDA 3rd 학술제 — 소비기한 추출 베이스라인 (최종본)

**과제**: 상품 이미지에서 소비기한 (Year, Month, Day) 추출
**평가**: Exact Match (Y·M·D 3개 값 모두 일치해야 정답)
**제공 데이터**: Train 상품 사진만 제공 (정답 라벨·좌표 라벨 모두 미제공)

---

## 이 베이스라인의 설계 방침

**순수 규칙 기반(Pure Rule-based)으로 단권화**했습니다.
Regex 파싱 + 방향성 앵커 스코어링만으로 동작하며, 별도의 학습 과정이 없어
**받는 즉시 실행 가능**합니다.

> **왜 ML을 베이스라인에서 뺐는가**
> 규칙 기반 점수로 Pseudo-label을 만든 뒤, *그 점수를 만든 것과 동일한 피처*로
> 모델을 학습시키면 데이터의 새로운 패턴을 배우는 게 아니라 사람이 정한 공식을
> 그대로 복제하게 됩니다(Self-Mimicry). 이는 순환 논리이며 성능 향상이 보장되지
> 않습니다. ML로 실질적 이득을 보려면 **규칙이 보지 못하는 새로운 정보원**
> (원본 픽셀, 문자 단위 인식 결과, 레이아웃 임베딩 등)을 피처로 넣어야 합니다.
> 구체적인 확장 방법은 노트북 마지막 **[ML 확장 가이드]** 섹션에 정리했습니다.

## 구조
```
PART 1.   환경 설정 & 전역 파라미터
MODULE 1. OCR 엔진 & 텍스트 정규화     : 적응형 리사이즈, 도트폰트 교정, BBox 병합
MODULE 2. 방향성 앵커 스코어링 & Regex : 상대 위치(Δx, Δy) 기반 지배력 판정
MODULE 3. 추론 파이프라인 & 제출 생성  : 폴백 처리, submission.csv
MODULE 4. 시각화 & 디버깅 툴           : 스코어링 과정을 눈으로 확인
```

---
## PART 1 — 환경 설정 & 전역 파라미터

In [ ]:
# ============================================================
# CELL 1 — 의존성 설치
# ============================================================
# [참가자 가이드] Colab 환경이 아닌 VS Code, Jupyter Notebook 환경에서
# Shell 명령어(!pip) 실행 시 패키지 인식 오류가 발생하면 %pip install ... 로 변경하세요.
!pip install -q easyocr opencv-python-headless pandas matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 (CPU 모드)"


In [ ]:
# ============================================================
# CELL 2 — Import & 전역 파라미터
# ============================================================
from __future__ import annotations

import os
import re
import glob
import time
import traceback
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
import easyocr

# ------------------------------------------------------------
# [설정] 본인 환경에 맞게 수정
# ------------------------------------------------------------
# [참가자 가이드] 아래 경로는 Google Colab 기본 구조 기준입니다.
# 로컬 PC(Windows/Mac)나 타 클라우드 환경 이용 시 본인의 데이터 폴더 경로로 반드시 변경하세요.
# 예: TRAIN_DIR = "./dataset/train_images" 또는 "C:/dataset/train_images"
TRAIN_DIR = "/content/dataset/train_images"   # 제공된 상품 사진 폴더
SUBMISSION_PATH = "submission.csv"

# --- 도메인 제약 ---
YEAR_MIN, YEAR_MAX = 2023, 2030

# --- 이미지 전처리 (속도/정확도 균형) ---
# [참가자 가이드] 실측 결과 원본 해상도(3024x4032) 그대로 OCR에 넣으면
# 장당 수십 초가 걸리고 메모리 초과로 프로세스가 죽기도 합니다.
# TARGET_MAX_SIDE를 낮추면 빨라지지만 작은 글씨를 놓칠 수 있습니다.
# 1280 기준 실측: 리사이즈 없이 35.4초 -> 적응형 10.9초 (3.2배 개선,
# 검출 텍스트 수는 오히려 26 -> 28개로 증가)
TARGET_MAX_SIDE = 1280    # 이보다 크면 축소
MIN_SIDE = 1000           # 이보다 작으면 확대 (최대 2배)

# --- 앵커 스코어링 ---
MAX_NORM_DIST = 12.0      # 글자높이 대비 이 배수 이상 떨어진 앵커는 무시
SAME_LINE_RATIO = 1.2     # |Δy| < 글자높이 * 이 값 이면 '같은 줄'로 간주
ABOVE_X_RATIO = 2.0       # 위쪽 앵커 인정 시 허용 가로 이탈 (세로 배치 라벨용)

USE_GPU = torch.cuda.is_available()
print(f"GPU 사용: {USE_GPU}")
print(f"이미지 폴더: {TRAIN_DIR}")
print(f"발견된 이미지 수: {len(glob.glob(os.path.join(TRAIN_DIR, '*.*')))}")

---
## MODULE 1 — OCR 엔진 & 텍스트 정규화

세 가지 문제를 처리합니다.
1. **속도**: 원본 해상도를 그대로 넣으면 병목 → 적응형 리사이즈
2. **도트 프린터 오인식**: `202S` → `2025`, `2O27` → `2027`
3. **조각 분리**: OCR이 `소비기한 / 06.26 / 까지`처럼 쪼개 인식 → 줄 단위 병합

In [ ]:
# ============================================================
# CELL 3 — [MODULE 1-1] 도트 폰트 / 공백 오인식 교정
# ============================================================

# 도트 프린터·잉크젯 각인에서 자주 발생하는 문자 혼동
DOT_FONT_CONFUSION = {
    'O': '0', 'o': '0', 'D': '0', 'Q': '0',
    'I': '1', 'l': '1', '|': '1', 'i': '1', '[': '1', ']': '1',
    'S': '5', 's': '5',
    'B': '8',
    'Z': '2', 'z': '2',
    'G': '6', 'b': '6',
    'q': '9', 'g': '9',
    'A': '4', 'T': '7',
}

# 구분자 오인식 (마침표가 쉼표·중점 등으로 읽히는 경우)
SEPARATOR_NORMALIZE = {'·': '.', '•': '.', '‧': '.', '․': '.', ',': '.'}

# 교정에서 보호할 단어 — 앵커 키워드가 깨지면 MODULE 2 스코어링이 무력화됨
# 예: 'LOT NO' -> 'L0T N0'이 되면 LOT 앵커를 못 찾음
PROTECTED_TOKENS = [
    'LOT', 'NO', 'EXP', 'MFG', 'BEST', 'BEFORE', 'USE', 'BY',
    'EXPIRY', 'DATE', 'BB', 'PROD',
]

_CHUNK_PATTERN = re.compile(r'[0-9OoDQIl|i\[\]SsBZzGbqgAT.\-/ ]{6,}')

# 덩어리 '맨 앞'에서부터 온전한 날짜가 성립하는지 판정 (.match로 사용)
# search가 아니라 match인 것이 핵심:
#   '2027.05.29 4' -> 맨 앞이 완전한 날짜 => 뒤의 ' 4'는 접미 문자이므로 건드리면 안 됨
#   '20 27.06.26'  -> 맨 앞은 '20 '이라 불성립 => 깨진 텍스트이므로 공백 복구 필요
# (search를 쓰면 후자에서도 내부의 '27.06.26'이 잡혀 복구가 실행되지 않음)
_HAS_FULL_DATE = re.compile(r'(?<!\d)\d{2,4}[.\-/]\d{1,2}[.\-/]\d{1,2}(?!\d)')


def _digit_ratio(token: str) -> float:
    core = [ch for ch in token if not ch.isspace() and ch not in '.-/,']
    return sum(ch.isdigit() for ch in core) / len(core) if core else 0.0


def clean_ocr_text(text: str, min_digit_ratio: float = 0.6) -> str:
    """
    OCR 오인식 교정. 3중 안전장치로 앵커 키워드 파괴를 방지한다.

      1) 보호 단어를 플레이스홀더로 치환 후 복원 (LOT, EXP 등)
      2) 숫자 비중이 min_digit_ratio 미만인 덩어리는 교정 제외
      3) 한글은 애초에 교정 대상 문자 집합에 없음

    [참가자 가이드]
      - 데이터를 직접 관찰하며 DOT_FONT_CONFUSION에 매핑을 추가하세요.
        (예: 특정 프린터에서 4가 A로, 7이 /로 읽히는 패턴)
      - min_digit_ratio를 낮추면 더 공격적으로 교정하지만 오교정 위험이 큽니다.
      - 교정 전/후를 MODULE 4 디버깅 셀로 반드시 비교하세요.
    """
    if not text:
        return text

    out = text
    for src, dst in SEPARATOR_NORMALIZE.items():
        out = out.replace(src, dst)

    # 1) 보호 단어 → 플레이스홀더
    placeholders: dict[str, str] = {}
    for idx, word in enumerate(PROTECTED_TOKENS):
        ph = f"\x00{idx}\x00"
        pat = re.compile(rf'\b{re.escape(word)}\b', re.IGNORECASE)
        if pat.search(out):
            out = pat.sub(ph, out)
            placeholders[ph] = word

    # 2) 날짜 후보 덩어리만 문자 교정
    def _fix(m: re.Match) -> str:
        chunk = m.group(0)
        if _digit_ratio(chunk) < min_digit_ratio:
            return chunk

        fixed = ''.join(DOT_FONT_CONFUSION.get(ch, ch) for ch in chunk)
        # 구분자 주변 공백 제거: "2027 . 06" -> "2027.06" (항상 안전)
        fixed = re.sub(r'\s*([.\-/])\s*', r'\1', fixed)

        # 숫자 사이 공백 제거는 '온전한 날짜가 아직 없을 때'만 수행한다.
        #
        # [이 조건이 없으면 생기는 버그 - 실측으로 발견]
        #   '2027.06.29 A' -> (A가 4로 교정) '2027.06.29 4'
        #   -> 무조건 공백을 지우면 '2027.06.294'가 되어 일(day) 자리가
        #      3자리로 망가지고 정규식이 통째로 실패한다.
        #   실제로 이 버그로 정상 인식되던 샘플이 NONE으로 퇴행했다.
        #
        # 따라서 맨 앞에서부터 날짜가 성립하면 손대지 않고, 깨진 경우
        # ('20 27.06.26' 처럼)에만 복구를 시도한다.
        if not _HAS_FULL_DATE.match(fixed.strip()):
            fixed = re.sub(r'(?<=\d)\s+(?=\d)', '', fixed)
        return fixed

    out = _CHUNK_PATTERN.sub(_fix, out)

    # 3) 보호 단어 복원
    for ph, word in placeholders.items():
        out = out.replace(ph, word)
    return out


# --- 자체 검증 (실행 시 바로 확인 가능) ---
_CLEAN_TESTS = [
    ('202S.06.26', '2025.06.26'),
    ('2O27.O6.26', '2027.06.26'),
    ('20 27.06.26', '2027.06.26'),
    ('2027 . 06 . 26', '2027.06.26'),
    ('2027,06,26', '2027.06.26'),
    ('LOT NO 543123', 'LOT NO 543123'),
    ('MFG 2026.03.10', 'MFG 2026.03.10'),
    ('소비기한 2027.06.26 까지', '소비기한 2027.06.26 까지'),
    ('2027.05.29 4', '2027.05.29 4'),   # 접미 문자를 일(day)에 붙이면 안 됨(회귀 방지)
]
_passed = sum(clean_ocr_text(src) == exp for src, exp in _CLEAN_TESTS)
print(f"clean_ocr_text 자체 검증: {_passed}/{len(_CLEAN_TESTS)} 통과")

In [ ]:
# ============================================================
# CELL 4 — [MODULE 1-2] OCR 엔진 (적응형 리사이즈)
# ============================================================

@dataclass
class OCRToken:
    """OCR이 인식한 텍스트 조각 하나."""
    bbox: list           # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    text: str            # 원본 인식 텍스트
    conf: float
    cleaned: str = ""    # clean_ocr_text 적용 결과

    @property
    def center(self) -> tuple[float, float]:
        xs = [p[0] for p in self.bbox]
        ys = [p[1] for p in self.bbox]
        return (sum(xs) / len(xs), sum(ys) / len(ys))

    @property
    def height(self) -> float:
        ys = [p[1] for p in self.bbox]
        return max(max(ys) - min(ys), 1.0)


class OCREngine:
    """EasyOCR 래퍼. 적응형 리사이즈로 속도/정확도를 균형 잡는다."""

    def __init__(self, langs: tuple[str, ...] = ('ko', 'en'), gpu: bool = USE_GPU):
        print("EasyOCR 모델 로딩 중... (최초 1회 다운로드 발생)")
        self.reader = easyocr.Reader(list(langs), gpu=gpu)
        print("로딩 완료")

    @staticmethod
    def preprocess(image_path: str | Path) -> np.ndarray:
        """
        적응형 리사이즈 + 그레이스케일.

        [설계 근거 - 실측]
          큰 이미지를 그대로 넣으면 느리고 메모리도 위험합니다.
          1512x2016 기준: 리사이즈 없음 35.4초 -> 적응형 10.9초 (3.2배)
          축소했는데도 검출 텍스트 수는 26 -> 28개로 오히려 늘었습니다.
          (EasyOCR 내부 검출기가 과대 해상도에서 오히려 분할을 놓치는 경향)

        [참가자 가이드]
          - 큰 이미지 축소엔 INTER_AREA, 작은 이미지 확대엔 INTER_CUBIC이
            일반적으로 유리합니다.
          - TARGET_MAX_SIDE를 올리면 작은 글씨엔 유리하나 느려집니다.
          - 곡면(병뚜껑)·원근 왜곡은 이 단계로 해결되지 않습니다.
            원근 변환(cv2.warpPerspective) 등을 추가로 검토하세요.
          - CLAHE/샤프닝은 실측에서 부작용(숫자 뭉개짐, 로트번호 오탐)이
            확인되어 기본값에서 제외했습니다. 추가 시 반드시 전후 비교하세요.
        """
        img = cv2.imread(str(image_path))
        if img is None:
            raise FileNotFoundError(f"이미지 로드 실패: {image_path}")

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        longest = max(h, w)

        if longest > TARGET_MAX_SIDE:
            scale = TARGET_MAX_SIDE / longest
            return cv2.resize(gray, (int(w * scale), int(h * scale)),
                              interpolation=cv2.INTER_AREA)
        if longest < MIN_SIDE:
            scale = min(2.0, MIN_SIDE / longest)
            return cv2.resize(gray, (int(w * scale), int(h * scale)),
                              interpolation=cv2.INTER_CUBIC)
        return gray

    def read(self, image_path: str | Path) -> list[OCRToken]:
        """OCR 실행 → OCRToken 리스트. 실패 시 빈 리스트 (파이프라인 중단 방지)."""
        try:
            img = self.preprocess(image_path)
            raw = self.reader.readtext(img)
        except Exception as exc:
            print(f"  [OCR 실패] {Path(image_path).name}: {exc}")
            return []

        tokens = []
        for bbox, text, conf in raw:
            tokens.append(OCRToken(bbox=bbox, text=text, conf=float(conf),
                                   cleaned=clean_ocr_text(text)))
        return tokens

    def read_array(self, image: np.ndarray) -> list[OCRToken]:
        """numpy 배열(크롭 등)에 직접 OCR. MODULE 3의 Custom Detector 경로용."""
        try:
            raw = self.reader.readtext(image)
        except Exception as exc:
            print(f"  [OCR 실패, array]: {exc}")
            return []
        return [OCRToken(bbox=b, text=t, conf=float(c), cleaned=clean_ocr_text(t))
                for b, t, c in raw]


ocr_engine = OCREngine()

In [ ]:
# ============================================================
# CELL 5 — [MODULE 1-3] 줄 단위 BBox 병합
# ============================================================

def merge_line_tokens(tokens: list[OCRToken],
                       y_tol_ratio: float = 0.6,
                       x_gap_ratio: float = 3.0) -> list[OCRToken]:
    """
    같은 줄에 있는 텍스트 조각을 이어붙인 토큰을 '추가'로 생성한다.
    (원본 조각도 그대로 유지 — 조각 단위 매칭 기회를 잃지 않기 위함)

    [왜 필요한가 — 실측 근거]
      OCR이 "소비기한 2027.06.26 까지"를 ["소비기한"], ["06.26"], ["까지"]처럼
      쪼개 인식하는 사례가 확인되었습니다. 연도와 월·일이 다른 조각으로 분리되면
      조각 단위 정규식으로는 절대 매칭되지 않습니다.

    임계값은 글자 높이 대비 상대값이라 해상도가 달라도 동작합니다.

    [참가자 가이드]
      y_tol_ratio를 키우면 더 공격적으로 병합합니다. 다만 무관한 텍스트까지
      엮여 오탐이 늘 수 있으니 MODULE 4로 병합 결과를 확인하며 조정하세요.
    """
    if not tokens:
        return []

    items = sorted(tokens, key=lambda t: t.center[1])
    used = [False] * len(items)
    merged: list[OCRToken] = list(items)

    for i, base in enumerate(items):
        if used[i]:
            continue
        group = [base]
        used[i] = True
        base_y, base_h = base.center[1], base.height

        for j in range(i + 1, len(items)):
            if used[j]:
                continue
            cand = items[j]
            if abs(cand.center[1] - base_y) > base_h * y_tol_ratio:
                continue
            if abs(cand.center[0] - group[-1].center[0]) > base_h * x_gap_ratio:
                continue
            group.append(cand)
            used[j] = True

        if len(group) < 2:
            continue

        group.sort(key=lambda t: t.center[0])
        text = ' '.join(t.text for t in group)
        pts = [p for t in group for p in t.bbox]
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        merged.append(OCRToken(
            bbox=[[min(xs), min(ys)], [max(xs), min(ys)],
                  [max(xs), max(ys)], [min(xs), max(ys)]],
            text=text,
            conf=sum(t.conf for t in group) / len(group),
            cleaned=clean_ocr_text(text),
        ))

    return merged

---
## MODULE 2 — 방향성 앵커 스코어링 & Regex 파싱

### 방향성 스코어링의 핵심 원리

**방향은 앵커의 부호를 바꾸지 않습니다. 방향이 결정하는 것은
"이 앵커가 이 날짜를 지배(govern)하는가"입니다.**

한국 상품 라벨의 전형적 배치를 보면:
```
제조일자  2026.03.10          <- negative 앵커도 자기 날짜의 왼쪽에 있음
소비기한  2027.06.26  까지     <- positive 앵커도 자기 날짜의 왼쪽에 있음
```
즉 `제조일자`와 `소비기한` **둘 다 왼쪽**에 옵니다. 따라서 왼쪽/위쪽 앵커는
부호가 +든 −든 **영향력을 증폭**해야 맞습니다.

**기존 유클리드 거리 방식의 오류**: 위 배치에서 `2027.06.26` 입장에선
`제조일자`가 대각선 위쪽에 가깝게 잡혀 −3.5점을 그대로 먹습니다. 하지만
`제조일자`가 실제로 지배하는 것은 윗줄의 `2026.03.10`입니다.

**실측 개선 효과** (위 2줄 배치 기준):

| 방식 | 제조일자 줄 | 소비기한 줄 | 변별력(격차) |
|---|---|---|---|
| 기존 유클리드 | +0.232 | +0.409 | 0.177 |
| **방향성 적용** | **−0.621** | **+1.281** | **1.901** |

변별력이 약 10배 커집니다.

In [ ]:
# ============================================================
# CELL 6 — [MODULE 2-1] Regex 파싱 & 도메인 제약 검증
# ============================================================

DATE_PATTERNS: list[tuple[str, re.Pattern]] = [
    ("YYYY_SEP", re.compile(r'(?<!\d)(\d{4})\s*[.\-/~]\s*(\d{1,2})\s*[.\-/~]\s*(\d{1,2})(?!\d)')),
    ("YY_SEP",   re.compile(r'(?<!\d)(\d{2})\s*[.\-/~]\s*(\d{1,2})\s*[.\-/~]\s*(\d{1,2})(?!\d)')),
    ("KOR",      re.compile(r'(?<!\d)(\d{4})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일')),
    ("KOR_YY",   re.compile(r'(?<!\d)(\d{2})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일')),
    ("YYYYMMDD", re.compile(r'(?<!\d)(\d{4})(\d{2})(\d{2})(?!\d)')),
    ("YYMMDD",   re.compile(r'(?<!\d)(\d{2})(\d{2})(\d{2})(?!\d)')),
]

# 날짜로 오인될 수 있는 숫자열을 '#'으로 마스킹 (길이 보존)
# (?<!\d)...(?!\d) 로 앞뒤 숫자 경계를 막는 것이 핵심.
# 이게 없으면 14자리 품목보고번호 '20130628332176' 안의 '20130628'이
# YYYYMMDD 패턴으로 걸려버린다.
NOISE_PATTERNS: list[re.Pattern] = [
    re.compile(r'(?<!\d)\d{13}(?!\d)'),                 # 바코드 EAN-13
    re.compile(r'(?<!\d)\d{14}(?!\d)'),                 # 품목보고번호
    re.compile(r'(?<!\d)\d{9,12}(?!\d)'),               # 기타 장문 코드
    re.compile(r'\d{2,4}\s*-\s*\d{3,4}\s*-\s*\d{4}'),   # 전화번호
]


def mask_noise(text: str) -> str:
    out = text
    for pattern in NOISE_PATTERNS:
        out = pattern.sub(lambda m: '#' * len(m.group(0)), out)
    return out


def validate_date(y: str, m: str, d: str) -> tuple[str, str, str] | None:
    """
    도메인 제약 검증. 통과 시 ('YYYY','MM','DD'), 실패 시 None.

      1) 2자리 연도 → 20XX 보정
      2) 연도 범위 (YEAR_MIN ~ YEAR_MAX)
      3) datetime 변환으로 실존 날짜 확인 (2월 30일 등 자동 배제)

    [참가자 가이드] 여기에 추가할 수 있는 제약:
      - 월/일 자리 뒤바뀜 복원 (2026.29.05 -> 2026.05.29)
      - 같은 이미지 내 제조일자와의 대소 관계 (소비기한 > 제조일자)
      - 촬영 시점 대비 상대 검증
    """
    try:
        yi, mi, di = int(y), int(m), int(d)
    except (ValueError, TypeError):
        return None

    if yi < 100:
        yi += 2000
    if not (YEAR_MIN <= yi <= YEAR_MAX):
        return None
    if not (1 <= mi <= 12 and 1 <= di <= 31):
        return None
    try:
        datetime(yi, mi, di)
    except ValueError:
        return None
    return (f"{yi:04d}", f"{mi:02d}", f"{di:02d}")


def parse_dates(text: str) -> list[dict[str, Any]]:
    """텍스트에서 유효한 날짜 후보를 모두 추출."""
    found: list[dict[str, Any]] = []
    masked = mask_noise(text)

    for name, pattern in DATE_PATTERNS:
        for match in pattern.finditer(masked):
            validated = validate_date(*match.groups())
            if validated:
                found.append({"ymd": validated, "pattern": name, "raw": match.group(0)})

    seen: set = set()
    unique: list[dict[str, Any]] = []
    for item in found:
        if item["ymd"] not in seen:
            seen.add(item["ymd"])
            unique.append(item)
    return unique


# --- 자체 검증 ---
_PARSE_TESTS = [
    ('소비기한 2027.06.26 까지', ('2027', '06', '26')),
    ('품목보고번호 20130628332176', None),   # 14자리 안의 8자리를 날짜로 오인하면 안 됨
    ('8801234567890', None),                 # 바코드
    ('소비자상담실 1899-1494', None),        # 전화번호
    ('2021.03.15', None),                    # 연도 범위 밖
    ('2026.02.30', None),                    # 실존하지 않는 날짜
    ('20260529', ('2026', '05', '29')),
    ('26.05.29', ('2026', '05', '29')),
]
_ok = sum((parse_dates(s)[0]['ymd'] if parse_dates(s) else None) == e
          for s, e in _PARSE_TESTS)
print(f"parse_dates 자체 검증: {_ok}/{len(_PARSE_TESTS)} 통과")

In [ ]:
# ============================================================
# CELL 7 — [MODULE 2-2] 방향성 앵커 스코어링
# ============================================================

# keyword: (weight, expected_position)
#   'prefix' = 날짜의 왼쪽/위쪽에 오는 것이 전형적 (라벨 역할)
#   'suffix' = 날짜의 오른쪽에 오는 것이 전형적 (조사 역할)
ANCHOR_SPECS: dict[str, tuple[float, str]] = {
    '소비기한':      (3.0,  'prefix'),
    '유통기한':      (3.0,  'prefix'),
    '품질유지기한':  (2.5,  'prefix'),
    'EXP':          (2.5,  'prefix'),
    'EXPIRY':       (2.5,  'prefix'),
    'BEST BEFORE':  (2.5,  'prefix'),
    'USE BY':       (2.5,  'prefix'),
    '까지':         (2.0,  'suffix'),
    '제조일':       (-3.5, 'prefix'),
    '제조':         (-3.0, 'prefix'),
    'MFG':          (-3.0, 'prefix'),
    'LOT':          (-2.5, 'prefix'),
    '품목보고번호':  (-3.0, 'prefix'),
    '상담실':       (-3.0, 'prefix'),
    '전화':         (-2.0, 'prefix'),
}


@dataclass
class Anchor:
    center: tuple[float, float]
    weight: float
    expect: str
    height: float
    keyword: str


def collect_anchors(tokens: list[OCRToken]) -> list[Anchor]:
    """OCR 토큰에서 앵커 키워드의 위치를 수집."""
    anchors: list[Anchor] = []
    for token in tokens:
        haystack = f"{token.text} {token.cleaned}".upper()
        for keyword, (weight, expect) in ANCHOR_SPECS.items():
            if keyword in token.text or keyword.upper() in haystack:
                anchors.append(Anchor(center=token.center, weight=weight,
                                      expect=expect, height=token.height,
                                      keyword=keyword))
    return anchors


def directional_weight(dx: float, dy: float, ref_height: float, expect: str) -> float:
    """
    앵커의 상대 위치로 '지배력 배수'를 반환.

      dx = anchor_x - date_x  (음수면 앵커가 날짜의 왼쪽)
      dy = anchor_y - date_y  (음수면 앵커가 날짜의 위쪽)

    반환값:
      1.5 = 전형적 배치. 이 앵커가 이 날짜를 지배할 확률 높음 → 영향력 증폭
      1.2 = 바로 위쪽(가로 정렬). 세로 배치 라벨
      0.3 = 배치가 어긋남. 다른 날짜를 지배할 가능성이 큼 → 영향력 축소
      0.5 = 그 외

    [참가자 가이드] 이 함수가 스코어링의 핵심입니다.
      - 배수(1.5/1.2/0.3/0.5)를 데이터로 튜닝해보세요.
      - 현재는 bbox 중심점만 씁니다. 좌우 경계(x_min, x_max)를 써서
        '가로 구간이 겹치는가'를 판정하면 더 정밀해집니다.
      - 표(table) 형태 라벨에서는 열(column) 정렬을 고려할 수 있습니다.
    """
    same_line = abs(dy) < ref_height * SAME_LINE_RATIO

    if expect == 'prefix':
        if same_line and dx < 0:
            return 1.5
        if (not same_line) and dy < 0 and abs(dx) < ref_height * ABOVE_X_RATIO:
            return 1.2   # 세로 배치. 대각선 위쪽은 여기 해당 안 됨(아래 0.5로)
        if same_line and dx > 0:
            return 0.3
        return 0.5

    # suffix
    if same_line and dx > 0:
        return 1.5
    if same_line and dx < 0:
        return 0.3
    return 0.5


def score_date_candidate(date_center: tuple[float, float],
                          date_height: float,
                          anchors: list[Anchor]) -> tuple[float, float, list[dict]]:
    """반환: (positive_score, negative_score, 진단용 상세)"""
    pos_total = neg_total = 0.0
    details: list[dict] = []

    for anchor in anchors:
        dx = anchor.center[0] - date_center[0]
        dy = anchor.center[1] - date_center[1]
        ref_h = max(date_height, anchor.height)

        norm_dist = ((dx ** 2 + dy ** 2) ** 0.5) / ref_h
        if norm_dist > MAX_NORM_DIST:
            continue

        dir_w = directional_weight(dx, dy, ref_h, anchor.expect)
        contribution = anchor.weight * dir_w * (1.0 / (1.0 + norm_dist))

        if anchor.weight > 0:
            pos_total += contribution
        else:
            neg_total += contribution

        details.append({
            'keyword': anchor.keyword, 'weight': anchor.weight,
            'expect': anchor.expect, 'dx': dx, 'dy': dy,
            'norm_dist': norm_dist, 'dir_w': dir_w, 'contribution': contribution,
        })

    return pos_total, neg_total, details


@dataclass
class DateCandidate:
    ymd: tuple[str, str, str]
    raw: str
    pattern: str
    score: float
    ocr_conf: float
    pos_score: float
    neg_score: float
    source_text: str
    anchor_details: list = field(default_factory=list)


def extract_candidates(tokens: list[OCRToken]) -> list[DateCandidate]:
    """
    OCR 토큰 → 점수순으로 정렬된 날짜 후보 리스트.

    최종 점수 = OCR신뢰도 + 포맷보너스 + 방향성 positive + 방향성 negative

    [참가자 가이드]
      점수 결합 방식(단순 합산)도 개선 여지가 큽니다. 가중 평균, 곱셈 결합,
      또는 후보 간 상대 비교(Reranking) 등을 시도해보세요.
    """
    merged = merge_line_tokens(tokens)
    anchors = collect_anchors(tokens)   # 앵커는 원본 토큰 기준으로 수집

    candidates: list[DateCandidate] = []
    for token in merged:
        for parsed in parse_dates(token.cleaned):
            pos, neg, details = score_date_candidate(token.center, token.height, anchors)
            format_bonus = 1.0 if parsed["pattern"] in ("YYYY_SEP", "KOR", "YYYYMMDD") else 0.0
            candidates.append(DateCandidate(
                ymd=parsed["ymd"], raw=parsed["raw"], pattern=parsed["pattern"],
                score=token.conf + format_bonus + pos + neg,
                ocr_conf=token.conf, pos_score=pos, neg_score=neg,
                source_text=token.text, anchor_details=details,
            ))

    # 동일 날짜는 최고점만 유지
    best: dict[tuple, DateCandidate] = {}
    for cand in candidates:
        if cand.ymd not in best or cand.score > best[cand.ymd].score:
            best[cand.ymd] = cand

    return sorted(best.values(), key=lambda c: c.score, reverse=True)

---
## MODULE 3 — 추론 파이프라인 & 제출 파일 생성

### ⚠️ Crop 사용 시 앵커 손실 주의

참가자가 YOLO 등으로 날짜 영역만 잘라내면 **주변의 `소비기한`, `까지` 같은
앵커 키워드가 잘려 나가 MODULE 2의 스코어링이 통째로 무력화**됩니다.
아래 `CustomDateDetector`는 이 문제를 세 가지 방식으로 다룹니다.

In [ ]:
# ============================================================
# CELL 8 — [MODULE 3-1] Custom Detector (선택 확장, 기본 비활성)
# ============================================================

# [참가자 가이드] 크롭 시 앵커 손실 방지 파라미터
CROP_PAD_RATIO_X = 1.5   # 검출 박스 너비의 이 배수만큼 좌우로 확장
CROP_PAD_RATIO_Y = 1.0   # 높이의 이 배수만큼 상하로 확장


class CustomDateDetector:
    """
    ★ 본인이 학습시킨 검출 모델을 여기에 연결하세요 (기본값: 비활성) ★

    라벨과 사전학습 검출모델은 제공되지 않습니다. 사용하려면
    makesense.ai 등으로 직접 라벨링 → YOLO 등으로 직접 학습해야 합니다.

    ┌──────────────────────────────────────────────────────────┐
    │ ⚠️ 크롭과 앵커 스코어링의 충돌 — 반드시 읽으세요           │
    ├──────────────────────────────────────────────────────────┤
    │ 날짜 영역만 타이트하게 crop하면 이런 일이 벌어집니다:      │
    │                                                          │
    │   원본:  소비기한  2027.06.26  까지                       │
    │   crop:            [2027.06.26]                          │
    │                                                          │
    │ '소비기한'과 '까지'가 잘려 나가 앵커가 0개가 되고,        │
    │ MODULE 2의 방향성 스코어링이 통째로 무력화됩니다.         │
    │ 그 결과 여러 날짜 후보 중 무엇이 소비기한인지 판별할       │
    │ 근거가 사라져, 오히려 성능이 떨어질 수 있습니다.          │
    │                                                          │
    │ 대응 전략 3가지 (아래 코드에 모두 구현):                  │
    │  (1) 넉넉한 padding — 박스 크기에 비례해 좌우로 크게 확장 │
    │      (한국 라벨은 앵커가 '왼쪽'에 오므로 특히 좌측 여유)   │
    │  (2) 전역 앵커 병용 — 크롭 OCR 결과 + 원본 전체에서 찾은  │
    │      앵커를 합쳐서 스코어링 (좌표계를 원본 기준으로 통일) │
    │  (3) 검출 실패 시 전체 OCR 폴백                           │
    └──────────────────────────────────────────────────────────┘
    """

    def __init__(self, weights_path: str | None = None,
                 conf_threshold: float = 0.25, target_class: str = 'date'):
        self.model = None
        self.conf_threshold = conf_threshold
        self.target_class = target_class

        if not weights_path or not os.path.exists(weights_path):
            print("[정보] Custom Detector 비활성 → 전체 이미지 OCR로 동작합니다.")
            return

        try:
            from ultralytics import YOLO
            self.model = YOLO(weights_path)
            names = list(self.model.names.values())
            if target_class not in names:
                print(f"[경고] '{target_class}' 클래스 없음. 실제 목록: {names}")
                self.model = None
            else:
                print(f"[정보] Custom Detector 활성화: {weights_path}")
        except Exception as exc:
            print(f"[경고] 모델 로드 실패 → 전체 OCR로 폴백합니다: {exc}")
            self.model = None

    def is_enabled(self) -> bool:
        return self.model is not None

    def detect(self, image_path: str | Path) -> list[tuple[int, int, int, int]]:
        """반환: [(x1,y1,x2,y2), ...] 신뢰도 높은 순."""
        if not self.is_enabled():
            return []
        try:
            results = self.model.predict(str(image_path),
                                          conf=self.conf_threshold, verbose=False)
        except Exception as exc:
            print(f"  [검출 실패] {Path(image_path).name}: {exc}")
            return []

        found = []
        for box in results[0].boxes:
            if self.model.names[int(box.cls[0])] != self.target_class:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            found.append((float(box.conf[0]), (x1, y1, x2, y2)))
        found.sort(key=lambda item: item[0], reverse=True)
        return [box for _, box in found]

    @staticmethod
    def expand_box(box: tuple[int, int, int, int],
                   img_shape: tuple[int, int]) -> tuple[int, int, int, int]:
        """
        앵커 손실 방지용 padding. 박스 크기에 '비례'해 확장한다
        (고정 픽셀값을 쓰면 해상도가 달라질 때 무의미해지므로).

        한국 상품 라벨은 앵커가 주로 왼쪽에 오므로 좌측을 더 넉넉히 잡는 것도
        방법입니다. 아래는 대칭 확장이며, 필요 시 pad_left를 키우세요.
        """
        x1, y1, x2, y2 = box
        h, w = img_shape[:2]
        bw, bh = x2 - x1, y2 - y1
        pad_x = int(bw * CROP_PAD_RATIO_X)
        pad_y = int(bh * CROP_PAD_RATIO_Y)
        return (max(0, x1 - pad_x), max(0, y1 - pad_y),
                min(w, x2 + pad_x), min(h, y2 + pad_y))


custom_detector = CustomDateDetector(weights_path=None)

In [ ]:
# ============================================================
# CELL 9 — [MODULE 3-2] 단일 이미지 추론
# ============================================================

NONE_RESULT: tuple[str, str, str] = ("NONE", "NONE", "NONE")


@dataclass
class PredictionDebug:
    ocr_count: int = 0
    merged_count: int = 0
    anchor_count: int = 0
    candidates: list = field(default_factory=list)
    used_detector: bool = False
    error: str | None = None


def predict_single(image_path: str | Path,
                    detector: CustomDateDetector = None,
                    return_debug: bool = False):
    """
    이미지 1장 → (year, month, day). 어떤 예외가 나도 NONE_RESULT로 폴백한다.

    Detector 활성 시 경로:
      검출 → padding 확장 crop → 크롭 OCR + 원본 전역 앵커 병용 → 스코어링
    """
    detector = detector if detector is not None else custom_detector
    debug = PredictionDebug(used_detector=bool(detector and detector.is_enabled()))

    try:
        tokens: list[OCRToken] = []

        if detector and detector.is_enabled():
            boxes = detector.detect(image_path)
            img = cv2.imread(str(image_path))

            if img is not None and boxes:
                for box in boxes:
                    x1, y1, x2, y2 = detector.expand_box(box, img.shape)
                    crop = img[y1:y2, x1:x2]
                    if crop.size == 0:
                        continue
                    for tok in ocr_engine.read_array(crop):
                        # 크롭 내 좌표 → 원본 좌표계로 복원 (좌표계 통일 필수)
                        tok.bbox = [[p[0] + x1, p[1] + y1] for p in tok.bbox]
                        tokens.append(tok)

            # 전략 (2): 크롭에서 앵커를 못 찾았으면 원본 전체에서 앵커를 보강
            if not collect_anchors(tokens):
                tokens.extend(ocr_engine.read(image_path))

        if not tokens:   # 전략 (3): 검출 실패/비활성 → 전체 OCR 폴백
            tokens = ocr_engine.read(image_path)

        debug.ocr_count = len(tokens)
        if not tokens:
            return (NONE_RESULT, debug) if return_debug else NONE_RESULT

        debug.anchor_count = len(collect_anchors(tokens))
        candidates = extract_candidates(tokens)
        debug.merged_count = len(merge_line_tokens(tokens))
        debug.candidates = candidates[:5]

        if not candidates:
            return (NONE_RESULT, debug) if return_debug else NONE_RESULT

        return (candidates[0].ymd, debug) if return_debug else candidates[0].ymd

    except Exception as exc:
        debug.error = f"{type(exc).__name__}: {exc}"
        print(f"  [예외] {Path(image_path).name} → {debug.error}")
        traceback.print_exc(limit=1)
        return (NONE_RESULT, debug) if return_debug else NONE_RESULT

In [ ]:
# ============================================================
# CELL 10 — [MODULE 3-3] 전체 추론 & submission.csv
# ============================================================

IMAGE_EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')


def list_images(directory: str | Path) -> list[str]:
    paths: list[str] = []
    for ext in IMAGE_EXTS:
        paths.extend(glob.glob(os.path.join(str(directory), ext)))
    return sorted(set(paths))


def run_inference(image_dir: str | Path,
                   output_path: str = SUBMISSION_PATH,
                   limit: int | None = None) -> pd.DataFrame | None:
    """
    폴더 전체 추론 → submission.csv (image_id, year, month, day)
    개별 이미지 실패는 NONE으로 기록하고 계속 진행한다.
    """
    paths = list_images(image_dir)
    if limit:
        paths = paths[:limit]
    if not paths:
        print(f"[경고] {image_dir} 에서 이미지를 찾지 못했습니다. 경로를 확인하세요.")
        return None

    print(f"총 {len(paths)}장 추론 시작 "
          f"(Detector: {'ON' if custom_detector.is_enabled() else 'OFF'})\n")

    rows: list[dict] = []
    extracted = 0
    started = time.time()

    for idx, path in enumerate(paths, 1):
        year, month, day = predict_single(path)
        if year != "NONE":
            extracted += 1
        rows.append({"image_id": Path(path).stem,
                     "year": year, "month": month, "day": day})

        if idx % 50 == 0 or idx == len(paths):
            elapsed = time.time() - started
            eta = elapsed / idx * (len(paths) - idx)
            print(f"  [{idx}/{len(paths)}] 추출 {extracted}건 "
                  f"({extracted / idx * 100:.1f}%) | "
                  f"경과 {elapsed:.0f}초 | 잔여 예상 {eta:.0f}초")

    total = time.time() - started
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n{'=' * 52}")
    print(f"완료: {output_path}")
    print(f"총 {len(df)}건 | 날짜 추출 {extracted}건 ({extracted / len(df) * 100:.1f}%)")
    print(f"전체 {total:.1f}초 | 평균 {total / len(df):.2f}초/장")
    print(f"{'=' * 52}")
    print("\n※ '추출률'은 정확도가 아닙니다. NONE이 아닌 값을 뱉은 비율일 뿐이며,")
    print("   그 값이 정답인지는 별개입니다.")
    return df

---
## MODULE 4 — 시각화 & 디버깅 툴

**성능 개선의 출발점은 "왜 틀렸는지"를 정확히 아는 것입니다.**
아래 도구로 실패를 두 유형으로 분류하세요.

- **인식 실패**: OCR이 글자 자체를 못 읽음 → MODULE 1(전처리) 문제
- **판별 실패**: 후보는 여럿인데 1위가 틀림 → MODULE 2(스코어링) 문제

In [ ]:
# ============================================================
# CELL 11 — [MODULE 4-1] 스코어링 과정 상세 출력
# ============================================================

def explain_prediction(image_path: str | Path, top_k: int = 3) -> None:
    """한 이미지의 후보별 점수 구성을 앵커 기여도까지 분해해서 보여준다."""
    (y, m, d), dbg = predict_single(image_path, return_debug=True)

    print("=" * 68)
    print(f"파일: {Path(image_path).name}")
    print(f"예측: {y}-{m}-{d}")
    print(f"OCR 토큰 {dbg.ocr_count}개 | 병합 후 {dbg.merged_count}개 | "
          f"앵커 {dbg.anchor_count}개 | Detector {'ON' if dbg.used_detector else 'OFF'}")
    if dbg.error:
        print(f"오류: {dbg.error}")

    if not dbg.candidates:
        print("\n날짜 후보 없음 → '인식 실패' 유형. MODULE 1(전처리)을 점검하세요.")
        print("=" * 68)
        return

    print(f"\n후보 상위 {min(top_k, len(dbg.candidates))}개:")
    for rank, cand in enumerate(dbg.candidates[:top_k], 1):
        mark = "★" if rank == 1 else " "
        print(f"\n {mark} [{rank}] {'-'.join(cand.ymd)}   총점 {cand.score:+.3f}")
        print(f"      = OCR신뢰도 {cand.ocr_conf:.3f}"
              f" + positive {cand.pos_score:+.3f}"
              f" + negative {cand.neg_score:+.3f}"
              f" (+포맷보너스)")
        print(f"      원문: '{cand.raw}'  |  패턴: {cand.pattern}")
        print(f"      출처 텍스트: '{cand.source_text[:60]}'")

        if cand.anchor_details:
            print("      앵커 기여 내역:")
            for det in sorted(cand.anchor_details,
                              key=lambda x: abs(x['contribution']), reverse=True)[:4]:
                side = "좌" if det['dx'] < 0 else "우"
                vert = "상" if det['dy'] < 0 else ("하" if det['dy'] > 0 else "동일")
                print(f"        '{det['keyword']}' w={det['weight']:+.1f} "
                      f"{side}/{vert} 거리{det['norm_dist']:.1f} "
                      f"방향배수{det['dir_w']:.1f} → {det['contribution']:+.3f}")

    if len(dbg.candidates) >= 2:
        margin = dbg.candidates[0].score - dbg.candidates[1].score
        print(f"\n 1위-2위 격차: {margin:.3f}", end="  ")
        print("(작을수록 판별이 애매 → 스코어링 개선 여지)" if margin < 1.0
              else "(충분히 변별됨)")
    print("=" * 68)


# 사용 예시
_samples = list_images(TRAIN_DIR)[:3]
for _p in _samples:
    explain_prediction(_p)

In [ ]:
# ============================================================
# CELL 12 — [MODULE 4-2] OCR 결과 시각화 (BBox + 교정 전후 비교)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def visualize_ocr(image_path: str | Path, show_all_boxes: bool = True) -> None:
    """
    이미지 위에 OCR BBox를 그리고, 앵커/날짜 후보를 색으로 구분한다.
      빨강 = negative 앵커, 초록 = positive 앵커, 파랑 = 날짜 후보
    """
    img = ocr_engine.preprocess(image_path)
    tokens = ocr_engine.read(image_path)
    anchors = collect_anchors(tokens)
    anchor_centers = {a.center: a for a in anchors}

    fig, ax = plt.subplots(figsize=(11, 14))
    ax.imshow(img, cmap='gray')

    for token in tokens:
        xs = [p[0] for p in token.bbox]
        ys = [p[1] for p in token.bbox]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)

        color, label = None, None
        if token.center in anchor_centers:
            anc = anchor_centers[token.center]
            color = 'lime' if anc.weight > 0 else 'red'
            label = anc.keyword
        elif parse_dates(token.cleaned):
            color, label = 'deepskyblue', parse_dates(token.cleaned)[0]['raw']
        elif show_all_boxes:
            color = 'gray'

        if color is None:
            continue
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor=color,
                                       linewidth=2 if label else 0.6))
        if label:
            ax.text(x1, y1 - 4, label, color=color, fontsize=9, weight='bold')

    ax.set_title(f"{Path(image_path).name}\n"
                 "초록=positive앵커  빨강=negative앵커  파랑=날짜후보")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    # 교정 전후 비교 (도트 폰트 교정이 실제로 작동했는지 확인)
    changed = [(t.text, t.cleaned) for t in tokens if t.text != t.cleaned]
    if changed:
        print("clean_ocr_text 교정 내역:")
        for before, after in changed[:15]:
            print(f"  '{before}'  →  '{after}'")
    else:
        print("교정된 텍스트 없음")


# 사용 예시 (필요할 때 주석 해제)
# visualize_ocr(list_images(TRAIN_DIR)[0])

In [ ]:
# ============================================================
# CELL 13 — 자체 점검 실행 & 제출 파일 생성
# ============================================================
# Validation 데이터는 제공되지 않습니다. 아래는 파이프라인 동작 확인용이며
# 실제 채점 점수가 아닙니다.
#
# [실제 채점 방식]
#   이 노트북의 코드를 predict.py로 정리해 GitHub 저장소에 제출하면,
#   주최측이 저장소를 clone하여 비공개 validation 데이터에 대해 동일한
#   코드로 직접 실행하고 채점합니다.

result_df = run_inference(TRAIN_DIR, "self_check.csv", limit=50)

if result_df is not None:
    display(result_df.head(10))
    none_cnt = (result_df["year"] == "NONE").sum()
    print(f"\nNONE 비율: {none_cnt}/{len(result_df)} "
          f"({none_cnt / len(result_df) * 100:.1f}%)")
    valid = result_df[result_df["year"] != "NONE"]
    if not valid.empty:
        print("\n연도 분포:")
        print(valid["year"].value_counts().sort_index())

---
# [참가자 가이드] 성능 개선 로드맵

## 1단계 — 실패 유형부터 분류하세요

MODULE 4의 `explain_prediction()`을 30~50장에 돌려 두 유형의 비율을 파악합니다.
**이 비율이 어디에 공수를 투입할지를 결정합니다.**

| 증상 | 유형 | 손볼 곳 |
|---|---|---|
| `날짜 후보 없음` | 인식 실패 | MODULE 1 — 전처리, 해상도, OCR 엔진 |
| 후보는 여럿인데 1위가 오답 | 판별 실패 | MODULE 2 — 앵커 가중치, 방향 배수 |
| 1·2위 격차가 1.0 미만 | 변별력 부족 | MODULE 2 — 피처 추가, 스코어 결합식 |

## 2단계 — 규칙 기반 내에서의 개선

| 위치 | 함수 | 개선 방향 |
|---|---|---|
| CELL 3 | `clean_ocr_text` | 데이터 관찰 후 혼동 매핑 추가 |
| CELL 4 | `preprocess` | 원근/곡면 보정, 적응형 대비, 해상도 튜닝 |
| CELL 5 | `merge_line_tokens` | 병합 임계값, 세로쓰기 대응 |
| CELL 6 | `validate_date` | 월/일 뒤바뀜 복원, 제조일자 대비 검증 |
| CELL 7 | `directional_weight` | 배수 튜닝, bbox 경계 기반 겹침 판정 |
| CELL 7 | `ANCHOR_SPECS` | 키워드 추가, 가중치 조정 |

## 3단계 — ML로 확장하려면 (중요)

**순환 논리를 피하세요.** 규칙 기반 점수로 만든 라벨을, 그 점수를 만든 것과
같은 피처로 학습시키면 사람이 정한 공식을 복제할 뿐입니다(Self-Mimicry).
성능이 오르지 않거나, 규칙의 편향까지 그대로 학습됩니다.

**ML이 실질적 이득을 주려면 규칙이 못 보는 새 정보원을 넣어야 합니다:**

- **시각 피처**: 후보 영역의 원본 픽셀 crop을 CNN에 통과시킨 임베딩
  (각인 방식·잉크 색·인쇄 위치 등 규칙이 표현 못 하는 정보)
- **문자 단위 신뢰도**: OCR의 per-character confidence, 후보 문자 분포
- **레이아웃 임베딩**: LayoutLM 계열로 텍스트+좌표를 함께 인코딩
- **상대 피처**: 같은 이미지 내 다른 후보 대비 상대 순위·거리 (Groupwise)

**추천 학습 방식 — Reranking**

한 이미지의 후보들을 하나의 그룹으로 묶어 순위를 학습시킵니다.
후보를 독립적으로 이진분류하는 것보다 이 문제에 훨씬 적합합니다.

- **Pairwise**: 같은 이미지의 (정답 후보, 오답 후보) 쌍을 만들어
  정답이 더 높은 점수를 받도록 학습 (RankNet, LambdaRank)
- **Groupwise**: `LGBMRanker`의 `lambdarank` objective로
  이미지를 group 단위로 묶어 학습

**라벨 확보 방법**: 직접 라벨링한 소량 데이터로 시작하거나, 규칙 기반
1·2위 격차가 매우 큰 샘플만 pseudo-label로 쓰되 **반드시 육안 검증**을
거치세요. 잘못된 라벨이 섞이면 오류가 고착화됩니다(Garbage In, Garbage Out).

## 4단계 — 검출 모델 도입 시 (CELL 8 주의사항 필독)

크롭하면 앵커가 잘려 스코어링이 무력화됩니다. `expand_box()`의 padding 비율을
충분히 주고, 크롭에서 앵커를 못 찾으면 원본 전역 앵커를 병용하도록
이미 구현해 두었습니다. 이 부분을 수정할 땐 반드시 MODULE 4로 앵커 개수가
0이 되지 않는지 확인하세요.

## 마지막 — 속도도 평가 대상입니다

정확도만 올리고 속도를 놓치지 마세요. 무거운 모델을 전 이미지에 돌리는 대신,
**경량 경로 우선 → 실패분만 무거운 경로로 폴백**하는 캐스케이드 구조가
평균 처리비용 측면에서 유리합니다.